# 04 Seeds of the selected policy, and the tuned Volt-VAR droop

**Seeds.** The selected candidate (horizon 144, 30-70% dead zone) was trained again with seeds 1 and 2, with exactly the same
settings. The reporting rule was written before these runs: every seed is shown, nothing is re-selected, and if a seed
fails (a) or (b) the use of active power is reported as not robust across seeds.

**Droop.** The original Volt-VAR droop setting was never tuned, while DPC was trained. To make the comparison fairer, the
droop was tuned once, on the **training part only**, over a pre-specified grid (round 1), and then over an extended grid
(round 2, added after seeing that the round-1 optimum lay at a corner of the grid). No tuning was done after round 2.

```{include} ../_caveats.md
```

In [ ]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from common import *
sd = load("m3_exp2_seeds.json")
display(Markdown(provenance(sd, "m3_exp2_seeds.json")))
rows = []
for seed, c in sd["candidates"].items():
    for reg, label in REGIMES.items():
        r = c["regimes"][reg]
        rows.append({"seed": seed.replace("seed_", ""), "regime": label, "objective": fmt(r["objective"]),
                     "mean |P| per battery (MW)": ", ".join(fmt(x) for x in r["mean_abs_p_mw"]),
                     "mean |Q| per battery (Mvar)": ", ".join(fmt(x, 3) for x in r["mean_abs_q_mvar"]),
                     "at S limit": pct(r["share_at_s_limit"], 1), "VVR": pct(r["summary"]["vvr_share_of_steps_with_any_violation"]),
                     "end-of-day SoC": f"{fmt(r['end_of_day_soc_min'], 2)} - {fmt(r['end_of_day_soc_max'], 2)}"})
table(rows).set_index(["seed", "regime"])

*Table: every seed of the selected candidate, both regimes. VVR = share of steps with any voltage violation.
"at S limit" = share of battery steps at the inverter apparent-power limit (apparent power at least 98% of the limit).*

In [ ]:
o = sd["over_seeds"]
rows = [{"seed": s.replace("seed_", ""), "(a) P used": fmt(sd["checks"][s]["uses_active_power"]),
         "(b) no drift": fmt(sd["checks"][s]["soc_stays_in_range"]), "both": fmt(sd["checks"][s]["passes"]),
         "min mean |P| (MW)": fmt(o["condition_a_minimum_per_seed"][s])} for s in sd["candidates"]]
display(table(rows, index="seed"))
display(Markdown(f"Seeds passing (a): **{o['n_pass_a']} of 3**, (b): **{o['n_pass_b']} of 3**, both: **{o['n_pass_both']} of 3**. "
                 f"The minimum mean |P| has median {fmt(o['condition_a_minimum_median'])} MW and range "
                 f"{fmt(o['condition_a_minimum_range'][0])}-{fmt(o['condition_a_minimum_range'][1])} MW against the 0.025 MW "
                 "threshold, so **the use of active power is not robust across seeds**."))

## Droop tuning on the training part (rounds 1 and 2)

In [ ]:
d1, d2 = load("m3_droop_tuning.json"), load("m3_droop_tuning_round2.json")
display(Markdown(provenance(d1, "m3_droop_tuning.json") + "  \n" + provenance(d2, "m3_droop_tuning_round2.json")))
settings = d1["training_part"] + d2["training_part_round2"]
bands = sorted({s["dead_band_pu"] for s in settings}); fulls = sorted({s["full_at_pu"] for s in settings})
grid = np.full((len(bands), len(fulls)), np.nan)
for s in settings:
    grid[bands.index(s["dead_band_pu"]), fulls.index(s["full_at_pu"])] = s["objective"]
fig, ax = plt.subplots(figsize=(7.5, 3.4))
im = ax.imshow(grid, cmap="Blues_r", aspect="auto")
ax.set_xticks(range(len(fulls)), [f"{f:g}" for f in fulls]); ax.set_yticks(range(len(bands)), [f"{b:g}" for b in bands])
ax.set_xlabel("deviation from 1 pu at which Q is at its limit (pu)"); ax.set_ylabel("dead band (pu)"); ax.grid(False)
for i in range(len(bands)):
    for j in range(len(fulls)):
        if not np.isnan(grid[i, j]):
            ax.text(j, i, f"{grid[i, j]:.4f}", ha="center", va="center", fontsize=8,
                    color="white" if grid[i, j] < np.nanmedian(grid) else INK)
ax.set_title("Training objective of each droop setting (lower is better; blank = not in either grid)", loc="left", fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8); plt.show()
f = d2["final_tuned"]
display(Markdown(f"Final tuned droop: dead band **{f['dead_band_pu']} pu**, full Q at **{f['full_at_pu']} pu** (round {f['round']}, "
                 f"training objective {fmt(f['training_objective'])}). Original: dead band {d2['original']['dead_band_pu']} pu, "
                 f"full Q at {d2['original']['full_at_pu']} pu."))

In [ ]:
best_r1 = min(s["objective"] for s in d1["training_part"])
best_r2 = min(s["objective"] for s in d2["training_part_round2"])
low_band = min(s["dead_band_pu"] for s in settings)
text = ("Round 2 did not find a better setting than round 1 " if best_r2 > best_r1 else "Round 2 found a better setting than round 1 ")
text += (f"(best training objective {best_r2:.5f} in round 2 against {best_r1:.5f} in round 1). ")
if f["dead_band_pu"] == low_band == 0:
    text += ("A dead band cannot be below 0, so the optimum at dead band 0 lies at a true boundary of that parameter. ")
text += "No tuning was done after round 2."
display(Markdown(text))
display(Markdown("*Note: the droop tuning objective above is computed on the training part, and the objectives below are "
                 "computed on the validation part. They come from different data and must not be compared with each other.*"))

## Validation: original droop, tuned droop and DPC (all seeds)

In [ ]:
names = {"droop_original": "droop, original", "droop_final_tuned": "droop, tuned (final)",
         "dpc_H144_dead_zone_30_70_seed_0": "DPC seed 0", "dpc_H144_dead_zone_30_70_seed_1": "DPC seed 1",
         "dpc_H144_dead_zone_30_70_seed_2": "DPC seed 2"}
rows = []
for key, label in names.items():
    v = d2["validation"][key]
    for reg, rl in REGIMES.items():
        r = v["regimes"][reg]
        rows.append({"controller": label, "regime": rl, "objective": fmt(r["objective"]),
                     "VVR": pct(r["vvr_share_of_steps_with_any_violation"], 3), "losses (kW)": fmt(r["mean_network_loss_kw"], 1),
                     "mean |Q| (Mvar)": ", ".join(fmt(x, 3) for x in r["mean_abs_q_mvar"]),
                     "Q sign changes": pct(r["share_q_sign_changes"], 1)})
display(table(rows).set_index(["controller", "regime"]))
display(Markdown("The tuned droop oscillates: its reactive power changes sign much more often than the original setting."))

*Table: validation part, both regimes. VVR = share of steps with any voltage violation.*

In [ ]:
s_lim = {reg: [sd["candidates"][s]["regimes"][reg]["share_at_s_limit"] for s in sd["candidates"]] for reg in REGIMES}
display(Markdown("The selected DPC seeds are at the inverter apparent-power limit on " +
                 " and ".join(f"{pct(min(v), 1)}-{pct(max(v), 1)} of battery steps in the {REGIMES[reg]} regime" for reg, v in s_lim.items()) +
                 ", so they use most of the inverter rating, mainly for reactive power."))
loss = {reg: ([d2["validation"][f"dpc_H144_dead_zone_30_70_seed_{i}"]["regimes"][reg]["mean_network_loss_kw"] for i in range(3)],
              d2["validation"]["droop_final_tuned"]["regimes"][reg]["mean_network_loss_kw"]) for reg in REGIMES}
dpc_b, droop_b = loss["base_x0.7"]
dpc_s, droop_s = loss["stress_x1.0"]
text = (f"Network losses: in the base regime DPC has {min(dpc_b):.1f}-{max(dpc_b):.1f} kW against {droop_b:.1f} kW for the "
        f"tuned droop")
if min(dpc_b) > droop_b:
    text += ", slightly higher, so there the objective advantage of DPC does not come from lower losses"
text += (f". In the stress regime DPC has {min(dpc_s):.1f}-{max(dpc_s):.1f} kW against {droop_s:.1f} kW for the tuned droop.")
display(Markdown(text))